# Lesson 5.1 — Health Rules

## Learning Objectives
- Explain what a health rule is and how it relates to BT performance
- Understand the built-in default health rules
- Create a custom health rule for a specific Business Transaction
- Configure health rule conditions using baselines and static values

**Reference**: [Health Rules](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond/health-rules) | [Alert and Respond](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond) | [Default Health Rules](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond/health-rules/default-health-rules)

## 1. What is a Health Rule?

A **Health Rule** defines what "healthy" means for a component. AppDynamics continuously evaluates these rules and changes the health status of tiers and BTs accordingly.

Think of it as a monitoring contract:
> *"If the /checkout transaction has an error rate above 5% AND at least 10 calls per minute, that's critical."*

### Health States:
| State | Indicator | Meaning |
|-------|-----------|--------|
| **Normal** | ✅ Green | All conditions within acceptable range |
| **Warning** | ⚠️ Yellow | Warning threshold breached |
| **Critical** | 🔴 Red | Critical threshold breached |
| **Unknown** | ⬜ Grey | Not enough data to evaluate |

Health states appear on:
- The **Flow Map** (tier box colors)
- The **Home** page application health summary
- **Policies** that trigger alerts when health changes

## 2. Default Health Rules

AppDynamics provides built-in health rules out of the box:

| Rule Name | Critical Condition | Warning Condition |
|-----------|--------------------|-------------------|
| **BT response time much higher than normal** | Avg RT > 3× baseline AND load > 50 cpm | Avg RT > 2× baseline AND load > 100 cpm |
| **BT error rate much higher than normal** | Error rate > 3× baseline AND > 10 errors/min AND load > 50 cpm | Error rate > 2× baseline AND > 5 errors/min AND load > 50 cpm |
| **CLR Garbage Collection Time too high** | GC > 30% of time spent | GC > 20% |
| **JVM Garbage Collection too high** | GC > 30% of time | GC > 20% |
| **Hardware CPU too high** | CPU > 95% | CPU > 85% |

These start working immediately after agent deployment — no configuration required.

**Reference**: [Default Health Rules](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond/health-rules/default-health-rules)

In [ ]:
# Model health rules as Python objects for learning

from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class Condition:
    metric: str
    operator: str  # GREATER_THAN, LESS_THAN, etc.
    value: float
    use_baseline: bool = False
    baseline_multiplier: float = 1.0

@dataclass
class HealthRule:
    name: str
    applies_to: str   # e.g., "BUSINESS_TRANSACTION", "JVM", "HARDWARE"
    scope: str        # e.g., "ALL" or specific BT name
    warning_conditions: List[Condition] = field(default_factory=list)
    critical_conditions: List[Condition] = field(default_factory=list)
    enabled: bool = True
    
    def evaluate(self, metrics: dict) -> str:
        """Evaluate health rule against current metric values."""
        def check(conditions):
            return all(self._check_condition(c, metrics) for c in conditions)
        if check(self.critical_conditions):
            return "CRITICAL"
        if check(self.warning_conditions):
            return "WARNING"
        return "NORMAL"
    
    def _check_condition(self, cond: Condition, metrics: dict) -> bool:
        value = metrics.get(cond.metric, 0)
        threshold = cond.value
        if cond.use_baseline and 'baseline_' + cond.metric in metrics:
            threshold = metrics['baseline_' + cond.metric] * cond.baseline_multiplier
        if cond.operator == 'GREATER_THAN':
            return value > threshold
        if cond.operator == 'LESS_THAN':
            return value < threshold
        return False

# Define the "BT response time much higher than normal" default rule
bt_response_rule = HealthRule(
    name="BT response time much higher than normal",
    applies_to="BUSINESS_TRANSACTION",
    scope="ALL",
    warning_conditions=[
        Condition("avg_response_time", "GREATER_THAN", 0, use_baseline=True, baseline_multiplier=2.0),
        Condition("calls_per_minute", "GREATER_THAN", 100),
    ],
    critical_conditions=[
        Condition("avg_response_time", "GREATER_THAN", 0, use_baseline=True, baseline_multiplier=3.0),
        Condition("calls_per_minute", "GREATER_THAN", 50),
    ]
)

# Test with different metric scenarios
scenarios = [
    {"name": "Normal traffic",    "avg_response_time": 280, "baseline_avg_response_time": 270, "calls_per_minute": 80},
    {"name": "Slight slowdown",   "avg_response_time": 620, "baseline_avg_response_time": 270, "calls_per_minute": 120},  # ~2.3× baseline
    {"name": "Major slowdown",    "avg_response_time": 1100, "baseline_avg_response_time": 270, "calls_per_minute": 75},  # ~4× baseline
    {"name": "Slow but low load", "avg_response_time": 1500, "baseline_avg_response_time": 270, "calls_per_minute": 3},   # slow but not enough load
]

print(f"Health Rule: {bt_response_rule.name}")
print()
for s in scenarios:
    name = s.pop('name')
    state = bt_response_rule.evaluate(s)
    icon = {"NORMAL": "✅", "WARNING": "⚠️ ", "CRITICAL": "🔴"}.get(state, "❓")
    rt = s['avg_response_time']
    baseline = s['baseline_avg_response_time']
    cpm = s['calls_per_minute']
    print(f"  {icon} {state:<10}  Scenario: {name}")
    print(f"            Avg RT={rt}ms ({rt/baseline:.1f}× baseline), CPM={cpm}")

## 3. Creating a Custom Health Rule

### Step-by-step in the Controller UI:

1. Navigate to **Alert & Respond → Health Rules**
2. Click **+ Add** (or **New Health Rule**)
3. Fill in:
   - **Name**: e.g., "Checkout SLA Violation"
   - **Affects**: Select "Business Transactions"
   - **Scope**: Choose "All Business Transactions" or specify specific BTs
4. Configure **Critical Condition**:
   - Click **Add Critical Condition**
   - For **BT Health Evaluation** conditions:
     - Select: Average Response Time
     - Condition: Greater Than
     - Value: 2000 (ms) — static threshold OR
     - Baseline: 3 standard deviations above the Default Baseline AND
     - Load: At least 10 calls per minute
5. Configure **Warning Condition** (optional, but recommended)
6. Enable the rule and **Save**

### Condition types:
| Condition | Description |
|-----------|-------------|
| **Average Response Time** | Mean RT for the BT |
| **Error Rate %** | Percentage of errored calls |
| **Calls per Minute** | Load guard (don't alert on 0 calls) |
| **95th Percentile RT** | Tail latency |
| **Number of Slow Calls** | Count of Slow-state transactions |

In [ ]:
# EXERCISE 5.1: Design health rules for the StoreFront application
#
# TASK: Design appropriate health rules for these SLAs:
#
# SLA 1: /checkout must complete in < 2 seconds, 99.5% success rate
# SLA 2: /search must complete in < 500ms for P95
# SLA 3: CPU on OrderService nodes must not exceed 80% for > 5 minutes

sla_rules = [
    {
        "sla": "SLA 1: /checkout < 2s, 99.5% success",
        "rule_name": "???",  # TODO
        "applies_to": "???",
        "scope": "???",      # specific BT or all?
        "warning": {
            "condition": "???",
            "threshold": "???",
            "load_guard": "???",  # min calls/min
        },
        "critical": {
            "condition": "???",
            "threshold": "???",
            "load_guard": "???",
        }
    },
    {
        "sla": "SLA 2: /search < 500ms P95",
        "rule_name": "???",
        "applies_to": "???",
        "scope": "???",
        "warning":  {"condition": "???", "threshold": "???"},
        "critical": {"condition": "???", "threshold": "???"}
    },
    {
        "sla": "SLA 3: OrderService CPU < 80%",
        "rule_name": "???",
        "applies_to": "???",  # Not a BT rule — what type?
        "scope": "OrderService tier",
        "warning":  {"condition": "CPU %Busy > ???%"},
        "critical": {"condition": "CPU %Busy > ???%"}
    }
]

print("Health Rule Designs:")
for r in sla_rules:
    print(f"\n  SLA: {r['sla']}")
    print(f"  Rule name: {r['rule_name']}")
    print(f"  Applies to: {r['applies_to']}")
    print(f"  Warning: {r['warning']}")
    print(f"  Critical: {r['critical']}")

## Summary

- **Health Rules** define what "normal" vs "warning" vs "critical" means for your application
- AppDynamics ships with **default health rules** that work out-of-the-box using dynamic baselines
- Health states (Normal/Warning/Critical) are reflected in **Flow Map colors** and **dashboards**
- Custom rules can use **static thresholds** (e.g., RT > 2000ms) or **baseline-relative conditions** (e.g., 3× standard deviations)
- Always add a **load guard** (minimum calls per minute) to prevent false alerts during low-traffic periods

---

**Next**: [Lesson 5.2 — Policies and Actions](lesson-5.2-policies-actions.ipynb)